# Stress Prediction — gender 제거를 다른 모델(LightGBM, RandomForest)에도 적용

v34 tree(ExtraTrees+pair)에서는 gender 제거가 CV MAE를 0.148033 -> 0.147650으로 개선시켰습니다.
이게 v34 구조에서만 통하는 건지, 다른 모델에도 일반적으로 적용되는 개선인지 확인합니다.

⚠️ 미리 말씀드리면: LightGBM/RandomForest는 gender 포함 상태에서 이미 CV MAE 0.19~0.22로
v34(0.148)보다 훨씬 나빴습니다. 피처 하나 제거로 이 격차가 메워질 가능성은 낮지만,
"방향성 자체"(gender 제거가 도움되는가 안 되는가)는 확인할 가치가 있습니다.


In [1]:

from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")


def add_core_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_core_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

categorical_cols = raw_X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
print("범주형 컬럼:", categorical_cols)


범주형 컬럼: ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level']


In [2]:

import lightgbm as lgb


def prep_for_lgbm(train_df, other_df, cat_cols):
    train_local = train_df.copy()
    other_local = other_df.copy()
    for col in cat_cols:
        train_local[col] = train_local[col].astype("category")
        other_local[col] = pd.Categorical(other_local[col], categories=train_local[col].cat.categories)
    return train_local, other_local


def lgbm_predict_fold(train_df, val_df, y_train, y_val, cat_cols):
    train_local, val_local = prep_for_lgbm(train_df, val_df, cat_cols)
    model = lgb.LGBMRegressor(
        objective="regression", n_estimators=3000, learning_rate=0.03,
        num_leaves=15, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.05, reg_lambda=0.05, random_state=RANDOM_SEED, verbosity=-1,
    )
    model.fit(
        train_local, y_train,
        eval_set=[(val_local, y_val)], eval_metric="mae",
        categorical_feature=cat_cols if cat_cols else "auto",
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
    )
    return model.predict(val_local, num_iteration=model.best_iteration_)


def rf_predict_fold(train_df, val_df, y_train, cat_cols):
    train_local = pd.get_dummies(train_df, columns=cat_cols, dummy_na=True)
    val_local = pd.get_dummies(val_df, columns=cat_cols, dummy_na=True)
    val_local = val_local.reindex(columns=train_local.columns, fill_value=0)
    model = RandomForestRegressor(
        n_estimators=800, max_features="sqrt", min_samples_leaf=5,
        random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_local, y_train)
    return model.predict(val_local)


## 5-Fold CV — gender 포함 vs 제외, LightGBM/RandomForest 각각

In [3]:

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)


def cv_mae(model_fn, drop_gender, use_lgbm):
    cols = [c for c in categorical_cols if not (drop_gender and c == "gender")]
    oof = np.zeros(len(y))
    for train_idx, val_idx in cv.split(raw_X):
        train_df = raw_X.iloc[train_idx].reset_index(drop=True)
        val_df = raw_X.iloc[val_idx].reset_index(drop=True)
        if drop_gender:
            train_df = train_df.drop(columns=["gender"])
            val_df = val_df.drop(columns=["gender"])
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].reset_index(drop=True)
        if use_lgbm:
            pred = lgbm_predict_fold(train_df, val_df, y_train, y_val, cols)
        else:
            pred = rf_predict_fold(train_df, val_df, y_train, cols)
        oof[val_idx] = pred
    return mean_absolute_error(y, oof)


rows = []
for model_name, use_lgbm in [("LightGBM", True), ("RandomForest", False)]:
    mae_with = cv_mae(None, drop_gender=False, use_lgbm=use_lgbm)
    mae_without = cv_mae(None, drop_gender=True, use_lgbm=use_lgbm)
    rows.append({
        "model": model_name, "gender_included_MAE": mae_with,
        "gender_removed_MAE": mae_without, "delta": mae_without - mae_with,
    })
    print(f"{model_name}: 포함={mae_with:.6f}, 제외={mae_without:.6f}, delta={mae_without - mae_with:+.6f}")

result_df = pd.DataFrame(rows)
print()
display(result_df)
print()
print("(참고) v34(ExtraTrees+pair) gender 제거 CV MAE: 0.147650 (원본 대비 -0.000383, v34에서만 유효했던 개선)")


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=

LightGBM: 포함=0.189706, 제외=0.189756, delta=+0.000050
RandomForest: 포함=0.223669, 제외=0.222910, delta=-0.000760



,model,gender_included_MAE,gender_removed_MAE,delta
0,LightGBM,0.189706,0.189756,0.00005
1,RandomForest,0.223669,0.222910,-0.00076



(참고) v34(ExtraTrees+pair) gender 제거 CV MAE: 0.147650 (원본 대비 -0.000383, v34에서만 유효했던 개선)
